In [0]:
import os, sys, importlib, datetime as dt
import pandas as pd

repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.append(os.path.join(repo_root, "src"))
import entsoe_parser
importlib.reload(entsoe_parser)
from entsoe_parser import parse

UTC = dt.timezone.utc
NS_PUB = "urn:iec62325.351:tc57wg16:451-3:publicationdocument:7:3"
NS_GL  = "urn:iec62325.351:tc57wg16:451-6:generationloaddocument:3:0"
NS_ACK = "urn:iec62325.351:tc57wg16:451-1:acknowledgementdocument:7:0"

# ---- Test 1: hourly day-ahead price, complete series (A01), 4 hours ----
t1 = f"""<Publication_MarketDocument xmlns="{NS_PUB}">
  <TimeSeries><mRID>1</mRID><businessType>A62</businessType>
    <in_Domain.mRID codingScheme="A01">10Y1001A1001A82H</in_Domain.mRID>
    <out_Domain.mRID codingScheme="A01">10Y1001A1001A82H</out_Domain.mRID>
    <currency_Unit.name>EUR</currency_Unit.name><price_Measure_Unit.name>MWH</price_Measure_Unit.name>
    <curveType>A01</curveType>
    <Period><timeInterval><start>2025-06-01T22:00Z</start><end>2025-06-02T02:00Z</end></timeInterval>
      <resolution>PT60M</resolution>
      <Point><position>1</position><price.amount>84.10</price.amount></Point>
      <Point><position>2</position><price.amount>79.55</price.amount></Point>
      <Point><position>3</position><price.amount>-5.20</price.amount></Point>
      <Point><position>4</position><price.amount>71.00</price.amount></Point>
    </Period></TimeSeries></Publication_MarketDocument>""".encode()

# ---- Test 2: 15-minute price with omitted points (A03), 1 hour ----
t2 = f"""<Publication_MarketDocument xmlns="{NS_PUB}">
  <TimeSeries><mRID>1</mRID><businessType>A62</businessType>
    <in_Domain.mRID codingScheme="A01">10YFR-RTE------C</in_Domain.mRID>
    <out_Domain.mRID codingScheme="A01">10YFR-RTE------C</out_Domain.mRID>
    <currency_Unit.name>EUR</currency_Unit.name><price_Measure_Unit.name>MWH</price_Measure_Unit.name>
    <curveType>A03</curveType>
    <Period><timeInterval><start>2025-10-05T10:00Z</start><end>2025-10-05T11:00Z</end></timeInterval>
      <resolution>PT15M</resolution>
      <Point><position>1</position><price.amount>12.00</price.amount></Point>
      <Point><position>3</position><price.amount>9.50</price.amount></Point>
    </Period></TimeSeries></Publication_MarketDocument>""".encode()

# ---- Test 3: generation by fuel incl. a consumption series (pumped storage) ----
t3 = f"""<GL_MarketDocument xmlns="{NS_GL}">
  <TimeSeries><mRID>1</mRID><businessType>A01</businessType>
    <inBiddingZone_Domain.mRID codingScheme="A01">10Y1001A1001A82H</inBiddingZone_Domain.mRID>
    <quantity_Measure_Unit.name>MAW</quantity_Measure_Unit.name><curveType>A01</curveType>
    <MktPSRType><psrType>B16</psrType></MktPSRType>
    <Period><timeInterval><start>2025-06-02T10:00Z</start><end>2025-06-02T12:00Z</end></timeInterval>
      <resolution>PT60M</resolution>
      <Point><position>1</position><quantity>41250</quantity></Point>
      <Point><position>2</position><quantity>43010</quantity></Point>
    </Period></TimeSeries>
  <TimeSeries><mRID>2</mRID><businessType>A04</businessType>
    <outBiddingZone_Domain.mRID codingScheme="A01">10Y1001A1001A82H</outBiddingZone_Domain.mRID>
    <quantity_Measure_Unit.name>MAW</quantity_Measure_Unit.name><curveType>A01</curveType>
    <MktPSRType><psrType>B10</psrType></MktPSRType>
    <Period><timeInterval><start>2025-06-02T10:00Z</start><end>2025-06-02T12:00Z</end></timeInterval>
      <resolution>PT60M</resolution>
      <Point><position>1</position><quantity>2100</quantity></Point>
      <Point><position>2</position><quantity>2350</quantity></Point>
    </Period></TimeSeries></GL_MarketDocument>""".encode()

# ---- Test 4: no-data acknowledgement ----
t4 = f"""<Acknowledgement_MarketDocument xmlns="{NS_ACK}">
  <mRID>x</mRID><Reason><code>999</code><text>No matching data found for Data item</text></Reason>
</Acknowledgement_MarketDocument>""".encode()

# ---- Test 5: cross-border physical flow FR -> ES, 2 hours ----
t5 = f"""<Publication_MarketDocument xmlns="{NS_PUB}">
  <TimeSeries><mRID>1</mRID><businessType>A66</businessType>
    <in_Domain.mRID codingScheme="A01">10YES-REE------0</in_Domain.mRID>
    <out_Domain.mRID codingScheme="A01">10YFR-RTE------C</out_Domain.mRID>
    <quantity_Measure_Unit.name>MAW</quantity_Measure_Unit.name><curveType>A01</curveType>
    <Period><timeInterval><start>2025-06-02T10:00Z</start><end>2025-06-02T12:00Z</end></timeInterval>
      <resolution>PT60M</resolution>
      <Point><position>1</position><quantity>2800</quantity></Point>
      <Point><position>2</position><quantity>3050</quantity></Point>
    </Period></TimeSeries></Publication_MarketDocument>""".encode()

# ---------------- Assertions ----------------
r1 = parse(t1)
assert len(r1) == 4
assert r1[0]["timestamp_utc"] == dt.datetime(2025, 6, 1, 22, 0, tzinfo=UTC)
assert r1[3]["timestamp_utc"] == dt.datetime(2025, 6, 2, 1, 0, tzinfo=UTC)
assert r1[2]["value"] == -5.20 and r1[0]["measure"] == "price" and r1[0]["currency"] == "EUR"
print("Test 1 passed: hourly prices, timestamps, negative price kept")

r2 = parse(t2)
assert len(r2) == 4, f"expected 4 quarter-hours, got {len(r2)}"
assert [r["value"] for r in r2] == [12.00, 12.00, 9.50, 9.50]
assert [r["is_filled"] for r in r2] == [False, True, False, True]
assert r2[1]["timestamp_utc"] == dt.datetime(2025, 10, 5, 10, 15, tzinfo=UTC)
print("Test 2 passed: 15-minute resolution, omitted A03 points filled and flagged")

r3 = parse(t3)
assert len(r3) == 4
solar = [r for r in r3 if r["psr_type"] == "B16"]
pump  = [r for r in r3 if r["psr_type"] == "B10"]
assert len(solar) == 2 and solar[0]["in_domain"] == "10Y1001A1001A82H" and solar[0]["out_domain"] is None
assert len(pump) == 2 and pump[0]["out_domain"] == "10Y1001A1001A82H" and pump[0]["in_domain"] is None
assert solar[0]["measure"] == "quantity" and solar[0]["unit"] == "MAW"
print("Test 3 passed: generation by fuel, consumption series identified by out zone")

assert parse(t4) == []
print("Test 4 passed: no-data acknowledgement returns no rows")

r5 = parse(t5)
assert len(r5) == 2 and r5[0]["out_domain"] == "10YFR-RTE------C" and r5[0]["in_domain"] == "10YES-REE------0"
print("Test 5 passed: flow direction FR -> ES captured")

print("\nAll parser tests passed.")
display(pd.DataFrame(r2)[["timestamp_utc", "resolution", "curve_type", "position", "value", "is_filled"]])